# AI Gaming Content Editor — Phase 0 pipeline

Run the cells top to bottom (tap ▶ on each). Increment 0.3 runs **S0 Preflight → S1 Stage-in → S2 Probe → S3 Master → S4 Analysis copy** and saves `media_info.json`, `master.mp4`, `analysis.mp4` and `analysis.wav` to **My Drive › AIEditor › work › <match_id>**. S3 is slow (roughly the length of the recording or longer) and there is no resume yet: if Colab disconnects, run C1–C5 again.

**Before the first run**
1. Upload the recording to **My Drive › AIEditor › inbox**.
2. The repo is public, so no GitHub token is needed. Only if it is made private: in Colab open **Secrets** (key icon), add `GITHUB_TOKEN` (fine-grained token, read-only *Contents* on this repo) and switch on *Notebook access*.

Cells: **C1** mount Drive · **C2** download the code at a tag · **C3** install PyYAML if missing · **C4** choose a recording · **C5** run S0 → S4 with live progress.

In [ ]:
#@title C1 · Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
#@title C2 · Download the pipeline code at a pinned tag
tag = "v0.3.0"  #@param {type:"string"}

import base64
import os
import subprocess
import sys

REPO_URL = "https://github.com/pjenil280505-max/ai-gaming-editor.git"
REPO_DIR = "/content/ai-gaming-editor"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # the repo is public; the secret is only needed if it becomes private

git = ["git"]
auth = None
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]


def sh(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.strip()
        for secret in (token, auth):
            if secret:
                message = message.replace(secret, "***")
        raise SystemExit(f"git failed: {message}")
    return result.stdout.strip()


if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(git + ["clone", "--quiet", REPO_URL, REPO_DIR])
sh(git + ["-C", REPO_DIR, "fetch", "--quiet", "--force", "--tags", "origin"])
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--force", tag])
sha = sh(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for name in [m for m in sys.modules if m == "core" or m.startswith("core.")]:
    del sys.modules[name]  # a new tag takes effect without restarting the runtime
print(f"Code ready: {tag} ({sha}) in {REPO_DIR}" + (" (using GITHUB_TOKEN)" if token else ""))

In [ ]:
#@title C3 · Install PyYAML if missing
import subprocess
import sys

try:
    import yaml
    print(f"PyYAML {yaml.__version__} already installed")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyYAML"], check=True)
    import yaml
    print(f"PyYAML {yaml.__version__} installed")

In [ ]:
#@title C4 · Choose a recording
#@markdown Leave **recording** empty and run this cell to list the inbox, then type a number or file name and run it again.
recording = ""  #@param {type:"string"}
target_fps = "config"  #@param ["config", "auto", "30", "60"]
force_rerun = False  #@param {type:"boolean"}

from core import run

print(run.describe_choice(recording) if recording.strip() else run.inbox_listing())

In [ ]:
#@title C5 · Run S0 → S4 with live progress
from core import run
from core.errors import StageError

try:
    chosen = recording.strip()
except NameError:
    chosen = ""
if not chosen:
    print("Choose a recording in cell C4 first.")
else:
    try:
        result = run.run_pipeline(chosen, target_fps=target_fps, force=force_rerun)
    except StageError as err:
        print(err)